In [1]:
# 감정분석 => 영화리뷰를 읽어서 긍정/부정인지 판별
import pandas as pd
train = pd.read_csv("http://114.207.245.181:13000/txt/ratings_train.txt", sep="\t")
test = pd.read_csv("http://114.207.245.181:13000/txt/ratings_test.txt", sep="\t")

train.tail(3)

,id,document,label
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1
149999,9619869,한국 영화 최초로 수간하는 내용이 담긴 영화,0


In [2]:
test.tail(3)

,id,document,label
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0
49999,6070594,마무리는 또 왜이래,0


In [42]:
import random
from itertools import islice

# na인것은 삭제
train_clean = train.dropna()
test_clean = test.dropna()

train_texts = train_clean['document'].to_list()
test_texts = test_clean['document'].to_list()

train_labels = train_clean['label'].to_list()
test_labels = test_clean['label'].to_list()

s = random.randint(0, len(train_texts) - 10)
for t,l in islice(zip(train_texts, train_labels), s, s + 10):
    print(f'긍정:{l}' if l == 1 else f'부정:{l}', t)

긍정:1 욕설 하나로 표현의 자유라는 주제를 이끌어내는 놀라운 다큐
긍정:1 좋은 영화 입니다. 강추
부정:0 이거 쿵푸팬더 따라한거 아녜요?!
긍정:1 4차원 드라마..그래도 재미있었다..
부정:0 잘나가다 또 .. 관객의 예상을 한치도 벗어나지 않는 전개 .
긍정:1 한국정부의 사과와 반성을 기대합니다
부정:0 [ 스토리 3, 비주얼 3, 연출 3, 연기 4 ] 잘 살렸다면 한국형 히어로로 성장할 수 있었을텐데.. 김수로 혼자선 역부족이다.
부정:0 굴곡진 인생길을 돌고 돌아 마침내 한 무대에 서게 된 세 사람.
부정:0 소재는 좋으나, 윤시윤 연기가 엉망!!!
긍정:1 내삶을 되돌아볼수있는 기회였고먹먹함과 울컥해지면서 나도모르게 흐르는 눈물많을걸생각하게 하는 영화였다많은분들의 희생으로 모든 직장인들에게행복한삶이 보장되길 바라며


In [43]:
def clean_texts(texts, labels):
    new_texts = []
    new_labels = []
    for t, l in zip(texts, labels):
        # text값이 비어있지 않고 공백을 제거했을때 최소 1자 이상인 것만 보관
        if t is not None and len(t.strip()) > 0:
            new_texts.append(t)
            new_labels.append(l)
    return new_texts, new_labels

train_texts1, train_labels1 = clean_texts(train_texts, train_labels)
test_texts1, test_labels1 = clean_texts(test_texts, test_labels)

In [44]:
print(len(train_texts1), len(train_labels))
print(len(test_texts1), len(test_labels))

149995 149995
49997 49997


In [45]:
import numpy as np
y_train = np.array(train_labels)
y_test = np.array(test_labels)

In [46]:
from transformers import AutoTokenizer

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
vocab_size = tokenizer.vocab_size

In [55]:
train_inputs = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    return_tensors="np",
    max_length=100,
)

test_inputs = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    return_tensors="np",
    max_length=100,
)


In [56]:
# 3. 단어 숫자 값의 길이를 맞춤 (pad)
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# import numpy as np

# x_train_pad1 = pad_sequences(x_train, maxlen=100, padding="post")
# x_test_pad1 = pad_sequences(x_test, maxlen=100, padding="post")

# x_train_pad = np.array(x_train_pad1)
# x_test_pad = np.array(x_test_pad1)

import tensorflow as tf
x_train = tf.convert_to_tensor(train_inputs["input_ids"])
x_test = tf.convert_to_tensor(test_inputs["input_ids"])
x_train_attention_mask = tf.convert_to_tensor(train_inputs["attention_mask"])
x_test_attention_mask = tf.convert_to_tensor(test_inputs["attention_mask"])
x_train.shape, x_test.shape, x_train_attention_mask.shape, x_test_attention_mask.shape

(TensorShape([149995, 100]),
 TensorShape([49997, 100]),
 TensorShape([149995, 100]),
 TensorShape([49997, 100]))

In [57]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

model = Sequential([
    Input(shape=(100,)),
     # mask_zero=True => 0의 값은 패딩으로처리
    Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True),
    LSTM(units=64, activation="tanh"),
    Dense(units=32, activation="relu"),
    Dense(units=1, activation="sigmoid")
])

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 100, 64)        │     2,048,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,083,137 (7.95 MB)

 Trainable params: 2,083,137 (7.95 MB)

 Non-trainable params: 0 (0.00 B)

In [58]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(
    x_train, y_train, validation_data=(x_test, y_test), epochs=1, verbose=2
)

In [ ]:

sample = "이 영화 진짜 재미 없다"
seq = tokenizer.texts_to_sequences([sample])
padded = pad_sequences(seq, maxlen=100, padding="post")

# 0 ~ 1
model.predict(padded)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


array([[0.00937823]], dtype=float32)